In [1]:
# integrantes:
# Daniela Costa da Silva 14613625
# Bruna Romero 11913896

In [2]:
#importando as bibliotecas

import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm
import ctypes
import math

In [3]:
#inicializando a janela

glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE);
window = glfw.create_window(720, 600, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)

In [4]:
#shaders 2D

#vertices
vertex_code = """
        attribute vec2 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,0.0,1.0);
        }
        """

#fragmentos
fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

In [5]:
#shaders 3D

#vertices
vertex_code_3d = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position, 1.0);
        }
        """

#fragmentos
fragment_code_3d = """
        uniform vec4 objectColor;
        void main(){
            gl_FragColor = objectColor;
        }
        """

In [6]:
# requisitando um programa e enviando slots para a GPU
#2D
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)


#3D
program_3d = glCreateProgram()
vertex_3d = glCreateShader(GL_VERTEX_SHADER)
fragment_3d = glCreateShader(GL_FRAGMENT_SHADER)

In [7]:
# Associando código aos shaders

#2D
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

#3D
glShaderSource(vertex_3d, vertex_code_3d)
glShaderSource(fragment_3d, fragment_code_3d)

In [8]:
#compilando o vertex shader

#2D
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader 2D")

#3D
glCompileShader(vertex_3d)
if not glGetShaderiv(vertex_3d, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex_3d).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader 3D")

In [9]:
#compilando o fragment shader

#2D
glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader 2D")

#3D
glCompileShader(fragment_3d)
if not glGetShaderiv(fragment_3d, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment_3d).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader 3D")

In [10]:
#inserindo os objetos no programa criado

#2D
glAttachShader(program, vertex)
glAttachShader(program, fragment)

#3D
glAttachShader(program_3d, vertex_3d)
glAttachShader(program_3d, fragment_3d)

In [11]:
#linkagem do programa

# Build program 2D
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking 2D error')

# Build program 2D
glLinkProgram(program_3d)
if not glGetProgramiv(program_3d, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program_3d))
    raise RuntimeError('Linking error 3D')
    
# Make program the default program
glUseProgram(program)

In [12]:
##parte do pipeline que vamos mexer 

# ----------------------------------------------------------- 2D ------------------------------------------------------------------------

#inserindo os dados dos vértices genérico que serão sobrescritos todas vez que desenhar um objeto 2D
# preparando espaço para 4 vértices usando 2 coordenadas (x,y)
vertices = np.zeros(4, [("position", np.float32, 2)])

# preenchendo as coordenadas de cada vértice

vertices['position'] = [
                            (+0.05, -0.05),
                            (+0.05, +0.05),
                            (-0.05, -0.05),
                            (-0.05, +0.05)
                        ]

# ----------------------------------------------------------- 3D ------------------------------------------------------------------------


#configurando buffers e criando a esfera
def criar_esfera(raio, segmentos):
    vertices = []
    indices = []

    for i in range(segmentos + 1):
        theta = i * math.pi / segmentos
        for j in range(segmentos + 1):
            phi = j * 2 * math.pi / segmentos
            
            x = raio * math.sin(theta) * math.cos(phi)
            y = raio * math.cos(theta)
            z = raio * math.sin(theta) * math.sin(phi)
            
            vertices.extend([x, y, z])
    
    for i in range(segmentos):
        for j in range(segmentos):
            a = i * (segmentos + 1) + j
            b = i * (segmentos + 1) + j + 1
            c = (i + 1) * (segmentos + 1) + j
            d = (i + 1) * (segmentos + 1) + j + 1
            
            indices.extend([a, b, c])
            indices.extend([b, d, c])
    
    return np.array(vertices, dtype=np.float32), np.array(indices, dtype=np.uint32)
    
#cria a bola
raio_bola = 0.08
segmentos = 16
vertices_bola, indices_bola = criar_esfera(raio_bola, segmentos)

#configurações 3D

# Criando VAO para a esfera 3D (vertex array object -> array de VBO's)
vao_bola = glGenVertexArrays(1)
glBindVertexArray(vao_bola)

# VBO para posições (vertex buffer object -> guarda dados dos vértices)
vbo_bola_pos = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, vbo_bola_pos)
glBufferData(GL_ARRAY_BUFFER, vertices_bola.nbytes, vertices_bola, GL_STATIC_DRAW)

# EBO para índices (elemnt buffer object -> guarda indice dos vertices)
ebo_bola = glGenBuffers(1)
glBindBuffer(GL_ELEMENT_ARRAY_BUFFER, ebo_bola)
glBufferData(GL_ELEMENT_ARRAY_BUFFER, indices_bola.nbytes, indices_bola, GL_STATIC_DRAW)

# Configurando atributos
loc_3d_pos = glGetAttribLocation(program_3d, "position")
glEnableVertexAttribArray(loc_3d_pos)
glVertexAttribPointer(loc_3d_pos, 3, GL_FLOAT, GL_FALSE, 0, None)

# Desvincula para segurança
glBindVertexArray(0)

In [13]:
# Inserindo elementos: Gato e Nuvem 3D
# O gato é formado por várias "caixas" e a nuvem reaproveita a mesma esfera (vao_bola) já criada, utilizando a mesma ideia dos arbustos.

def gerar_caixa(cx, cy, cz, sx, sy, sz):
# Gera os triângulos (36 vértices) de uma caixa centrada em (cx,cy,cz) com dimensões (sx,sy,sz) para gerar o corpo do gato
    hx, hy, hz = sx / 2, sy / 2, sz / 2
    v = {
        0: (cx - hx, cy - hy, cz - hz), 1: (cx + hx, cy - hy, cz - hz),
        2: (cx + hx, cy + hy, cz - hz), 3: (cx - hx, cy + hy, cz - hz),
        4: (cx - hx, cy - hy, cz + hz), 5: (cx + hx, cy - hy, cz + hz),
        6: (cx + hx, cy + hy, cz + hz), 7: (cx - hx, cy + hy, cz + hz),
    }
    faces = [
        (0, 1, 2), (0, 2, 3),   # face de trás
        (4, 6, 5), (4, 7, 6),   # face da frente
        (0, 4, 5), (0, 5, 1),   # face de baixo
        (3, 2, 6), (3, 6, 7),   # face de cima
        (0, 3, 7), (0, 7, 4),   # face esquerda
        (1, 5, 6), (1, 6, 2),   # face direita
    ]
    vertices_caixa = []
    for face in faces:
        for idx in face:
            vertices_caixa.append(v[idx])
    return vertices_caixa


def gerar_gato_3d():
# Monta o corpo do gato com várias caixas
    vertices_gato = []
    vertices_gato += gerar_caixa(0.00, 0.00, 0.00, 0.16, 0.07, 0.06)      # corpo
    vertices_gato += gerar_caixa(0.10, 0.035, 0.00, 0.07, 0.07, 0.06)     # cabeça
    vertices_gato += gerar_caixa(0.07, 0.075, -0.015, 0.02, 0.03, 0.02)   # orelha esquerda
    vertices_gato += gerar_caixa(0.07, 0.075, 0.015, 0.02, 0.03, 0.02)    # orelha direita
    vertices_gato += gerar_caixa(-0.10, 0.01, 0.00, 0.05, 0.02, 0.02)     # atrás (base)
    vertices_gato += gerar_caixa(-0.15, 0.04, 0.00, 0.05, 0.02, 0.02)     # atrás (ponta, mais alta)
    vertices_gato += gerar_caixa(0.06, -0.05, -0.02, 0.025, 0.05, 0.025)  # pata dianteira esquerda
    vertices_gato += gerar_caixa(0.06, -0.05, 0.02, 0.025, 0.05, 0.025)   # pata dianteira direita
    vertices_gato += gerar_caixa(-0.06, -0.05, -0.02, 0.025, 0.05, 0.025) # pata traseira esquerda
    vertices_gato += gerar_caixa(-0.06, -0.05, 0.02, 0.025, 0.05, 0.025)  # pata traseira direita
    return vertices_gato


# Gerando os dados do gato
lista_gato = gerar_gato_3d()
vertices_gato = np.zeros(len(lista_gato), [("position", np.float32, 3)])
vertices_gato['position'] = lista_gato
qtd_vertices_gato = len(lista_gato)

# VAO/VBO do gato (mandando os buffers de vertice e array de vertice para a GPU)
vao_gato = glGenVertexArrays(1)
glBindVertexArray(vao_gato)
vbo_gato = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, vbo_gato)
glBufferData(GL_ARRAY_BUFFER, vertices_gato.nbytes, vertices_gato, GL_STATIC_DRAW)
glEnableVertexAttribArray(loc_3d_pos)
glVertexAttribPointer(loc_3d_pos, 3, GL_FLOAT, GL_FALSE, 0, None)
glBindVertexArray(0)


# A nuvem reaproveita a esfera (vao_bola / ebo_bola) 
puffs_nuvem = [
    (0.00, 0.00, 0.00, 1.00),     # esfera central
    (-0.09, -0.015, 0.05, 0.75),  # esfera esquerdo
    (0.09, -0.015, -0.05, 0.75),   # esfera direito
    (-0.045, 0.035, -0.03, 0.65),  # esfera superior esquerdo
    (0.045, 0.035, 0.03, 0.65),  # esfera superior direito
]

In [14]:
#requisitando slot para enviar os dados da CPU para a GPU

#2D
# Request a buffer slot from GPU
buffer_VBO = glGenBuffers(1)
# Make this buffer the default one
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)



In [15]:
#enviando o conteúdo dos vértices

# Upload data
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

In [16]:
#Associando variáveis do programa GLSL (Vertex Shader) com nossos dados
#Associando atributos do shader
# --------------------------------------
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)

loc = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc)

glVertexAttribPointer(loc, 2, GL_FLOAT, False, stride, offset)

loc_color = glGetUniformLocation(program, "color")
R = 0.7
G = 0.0
B = 0.2
loc_transform = glGetUniformLocation(program, "mat_transformation")

In [17]:
#funções de desenho dos objetos na cena (ainda em 2D)

def desenhar_retangulo(x,y,largura,altura,cor):
    ###desenha um retângulo com as coordenadas (x,y) variando a altura e a largura)"""
    vertices = np.zeros(4, [("position", np.float32, 2)])
    vertices['position'] = [
        (x,y),                      #vertice 0
        (x + largura, y),            #vertice 1
        (x, y + altura),             #vertice 2
        (x + largura, y + altura)   #vertice 3
    ]

    #enviar para a GPU
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)
    glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
    
    # Define a cor
    glUniform4f(loc_color, cor[0], cor[1], cor[2], 1.0)
    
    # Desenha
    glDrawArrays(GL_TRIANGLE_STRIP, 0, 4)


def desenhar_triangulo(x1, y1, x2, y2, x3, y3, cor):
    """desenha um triangulo com 3 vértices"""
    vertices = np.zeros(3, [("position", np.float32, 2)])
    vertices['position'] = [
        (x1, y1),
        (x2, y2),
        (x3, y3)
    ]


    #mandando para a GPU
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)
    glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
    
    glUniform4f(loc_color, cor[0], cor[1], cor[2], 1.0)
    glDrawArrays(GL_TRIANGLES, 0, 3)


def desenhar_linha(x1, y1, x2, y2, cor, espessura = 1.0):
    """desenha uma linha entre dois pontos"""
    vertices = np.zeros(2, [("position", np.float32, 2)])
    vertices['position'] = [
        (x1, y1),
        (x2, y2)
    ]
    
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)
    glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
    
    glUniform4f(loc_color, cor[0], cor[1], cor[2], 1.0)
    glDrawArrays(GL_LINES, 0, 2)


def desenhar_arvore(x, y , cot_tronco, cor_copa):
    """Desenha uma árvore na posição (x,y)"""
    # Tronco (retângulo)
    desenhar_retangulo(x - 0.03, y, 0.06, 0.15, cor_tronco)
    
    # Copa (3 círculos/retângulos para simular)
    # Copa inferior (vamos usar retângulo)
    desenhar_retangulo(x - 0.15, y + 0.12, 0.30, 0.12, cor_copa)
    desenhar_retangulo(x - 0.12, y + 0.20, 0.24, 0.12, cor_copa)
    desenhar_retangulo(x - 0.09, y + 0.28, 0.18, 0.10, cor_copa)


def desenhar_cruz(x, y, tamanho, cor):
    """Desenha uma cruz (linhas horizontal e vertical)"""
    """usaremos para desenhar as janelas"""
    
    metade = tamanho / 2.0
    
    # Linha horizontal
    desenhar_linha(x - metade, y, x + metade, y, cor)
    
    # Linha vertical
    desenhar_linha(x, y - metade, x, y + metade, cor)


def desenhar_casa(x, y):
    """desenha uma casa na posição (x,y) usando as funções de formas básicas definidas acima"""

    #parede
    desenhar_retangulo(x - 0.25, y, 0.50, 0.30, (0.9, 0.8, 0.6))  # Bege

    #telhado
    desenhar_triangulo(
        x - 0.30, y + 0.30,  # vértice esquerdo
        x + 0.30, y + 0.30,  # vértice direito
        x, y + 0.50,         # vértice topo
        (0.6, 0.2, 0.1)      # Marrom
    )

    #divisão do telhado com uma linha
    desenhar_linha(x - 0.17, y + 0.38, x + 0.17, y + 0.38, (0.4, 0.2, 0.1))
    desenhar_linha(x, y + 0.30, x, y + 0.48, (0.4, 0.2, 0.1)) #vertical

    #porta
    desenhar_retangulo(x - 0.06, y, 0.12, 0.16, (0.5, 0.3, 0.1))  # Marrom escuro

    #maçaneta da porta
    desenhar_retangulo(x + 0.03, y + 0.06, 0.015, 0.015, (1.0, 0.8, 0.0))  # Amarelo

    #janela
    janela_x = x - 0.15
    janela_y = y + 0.08
    janela_tam = 0.08
    
    # Moldura da janela (são duas)
    desenhar_retangulo(janela_x - janela_tam/2, janela_y - janela_tam/2, 
                       janela_tam, janela_tam, (0.8, 0.8, 0.9))  # Azul claro
    
    # Cruz da janela
    desenhar_cruz(janela_x, janela_y, janela_tam * 0.9, (0.3, 0.3, 0.3))  # Cinza escuro

    #segunda
    janela_x = x + 0.15
    desenhar_retangulo(janela_x - janela_tam/2, janela_y - janela_tam/2, 
                       janela_tam, janela_tam, (0.8, 0.8, 0.9))
    desenhar_cruz(janela_x, janela_y, janela_tam * 0.9, (0.3, 0.3, 0.3))

def desenhar_baloes(x, y):
    """Desenha balões na posição (x,y) - todos em 2D"""
    
    cores_baloes = [
        (0.9, 0.1, 0.1),  # Vermelho
        (0.1, 0.8, 0.1),  # Verde
        (0.1, 0.4, 0.9),  # Azul
        (0.9, 0.9, 0.1),  # Amarelo
        (0.9, 0.5, 0.1),  # Laranja
        (0.8, 0.1, 0.8),  # Roxo
        (0.1, 0.8, 0.8),  # Ciano
        (0.9, 0.2, 0.4),  # Rosa
    ]
    
    raio_balao = 0.035
    espacamento = 0.085
    
    # Fios dos balões (linhas verticais)
    for i, cor in enumerate(cores_baloes):
        # Posição do balão em círculo
        angulo = 2 * math.pi * i / len(cores_baloes)
        bx = x + 0.12 * math.cos(angulo)
        by = y + 0.08 * math.sin(angulo) + 0.12
        
        # Linha do fio (do balão até a casa)
        desenhar_linha(bx, by - raio_balao, bx, y + 0.05, (0.3, 0.3, 0.3), espessura=0.5)
    
    # Desenha cada balão
    for i, cor in enumerate(cores_baloes):
        angulo = 2 * math.pi * i / len(cores_baloes)
        bx = x + 0.12 * math.cos(angulo)
        by = y + 0.08 * math.sin(angulo) + 0.12
        
        # Corpo do balão (círculo usando um pequeno quadrado com cantos arredondados)
        # Vamos usar um retângulo pequeno para simular
        desenhar_retangulo(bx - raio_balao, by - raio_balao, 
                          raio_balao * 2, raio_balao * 2, cor)
        
        # Brilho do balão (pequeno ponto branco)
        desenhar_retangulo(bx - raio_balao*0.2, by + raio_balao*0.3, 
                          raio_balao*0.15, raio_balao*0.15, (1.0, 1.0, 1.0))


def desenhar_carro(x, y, escala):
    """Desenha um carro simples na posição (x,y) usando retângulos que aumenta 
       ou diminui quando a tecla espaço é clicada"""

    # Aplica a escala em todas as dimensões
    largura = 0.30 * escala
    altura = 0.08 * escala
    
    # Corpo principal do carro
    desenhar_retangulo(x - largura/2, y, largura, altura, (0.2, 0.4, 0.8))  # Azul escuro
    
    # Cabine (parte de cima)
    cabine_largura = 0.12 * escala
    cabine_altura = 0.06 * escala
    desenhar_retangulo(x - cabine_largura/2, y + altura, cabine_largura, cabine_altura, (0.6, 0.8, 0.9))  # Azul claro
    
    # Para-choque dianteiro
    parachoques_largura = 0.03 * escala
    parachoques_altura = 0.06 * escala
    desenhar_retangulo(x + largura/2, y + 0.01 * escala, parachoques_largura, parachoques_altura, (0.8, 0.8, 0.8))  # Cinza
    
    # Para-choque traseiro
    desenhar_retangulo(x - largura/2 - parachoques_largura, y + 0.01 * escala, parachoques_largura, parachoques_altura, (0.8, 0.8, 0.8))  # Cinza
    
    # Rodas (usando retângulos pequenos)
    roda_tam = 0.04 * escala
    # Roda traseira esquerda
    desenhar_retangulo(x - 0.12 * escala, y - 0.025 * escala, roda_tam, roda_tam, (0.1, 0.1, 0.1))  # Preto
    # Roda traseira direita
    desenhar_retangulo(x + 0.08 * escala, y - 0.025 * escala, roda_tam, roda_tam, (0.1, 0.1, 0.1))  # Preto
    
    # Faróis
    farol_tam = 0.015 * escala
    desenhar_retangulo(x + largura/2, y + 0.04 * escala, farol_tam, farol_tam, (1.0, 1.0, 0.5))  # Amarelo claro
    desenhar_retangulo(x + largura/2, y - 0.01 * escala, farol_tam, farol_tam, (1.0, 1.0, 0.5))  # Amarelo claro
    
    # Lanternas traseiras
    lanterna_tam = 0.015 * escala
    desenhar_retangulo(x - largura/2 - parachoques_largura, y + 0.04 * escala, lanterna_tam, lanterna_tam, (1.0, 0.2, 0.2))  # Vermelho
    desenhar_retangulo(x - largura/2 - parachoques_largura, y - 0.01 * escala, lanterna_tam, lanterna_tam, (1.0, 0.2, 0.2))  # Vermelho
    
    # Janelas (pequenos detalhes)
    janela_tam = 0.03 * escala
    desenhar_retangulo(x - 0.04 * escala, y + altura + 0.01 * escala, janela_tam, janela_tam, (0.7, 0.8, 0.9))
    desenhar_retangulo(x + 0.01 * escala, y + altura + 0.01 * escala, janela_tam, janela_tam, (0.7, 0.8, 0.9))
    
    # Detalhe do para-choque
    desenhar_linha(x - largura/2, y - 0.005 * escala, x + largura/2, y - 0.005 * escala, (0.3, 0.3, 0.3))

def desenhar_circulo(cx, cy, raio, cor, segmentos=36):
    """Desenha um círculo aproximado usando triângulos"""
    vertices = []

    #centro do circulo
    vertices.append((cx, cy))

    #vertices ao redor do centro
    for i in range(segmentos +1 ):
        angulo = (2 * math.pi * i) / segmentos
        x = cx + raio * math.cos(angulo)
        y = cy + raio * math.sin(angulo)
        vertices.append((x,y))

    # Criar array de vértices para GL_TRIANGLE_FAN
    vertices_array = np.zeros(len(vertices), [("position", np.float32, 2)])
    vertices_array['position'] = vertices
    
    # Enviar para GPU
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)
    glBufferData(GL_ARRAY_BUFFER, vertices_array.nbytes, vertices_array, GL_DYNAMIC_DRAW)
    
    # Define a cor
    glUniform4f(loc_color, cor[0], cor[1], cor[2], 1.0)
    
    # Desenha como triângulos em leque
    glDrawArrays(GL_TRIANGLE_FAN, 0, len(vertices))


def desenhar_arbusto_com_flores(x, y, escala):
    """Desenha um arbusto com flores na posição (x,y)"""

    # Arbusto principal (vários círculos verdes sobrepostos)
    raio_arbusto = 0.06 * escala
    
    # Arbusto base (3 círculos verdes)
    desenhar_circulo(x, y, raio_arbusto, verde_arbusto)
    desenhar_circulo(x - raio_arbusto*0.8, y - raio_arbusto*0.3, raio_arbusto*0.9, verde_claro)
    desenhar_circulo(x + raio_arbusto*0.8, y - raio_arbusto*0.3, raio_arbusto*0.9, verde_claro)
    desenhar_circulo(x - raio_arbusto*0.4, y + raio_arbusto*0.5, raio_arbusto*0.8, verde_claro)
    desenhar_circulo(x + raio_arbusto*0.4, y + raio_arbusto*0.5, raio_arbusto*0.7, verde_claro)
    desenhar_circulo(x, y + raio_arbusto*0.6, raio_arbusto*0.8, verde_claro)
    
    # Flores (círculos menores com cores vibrantes)
    raio_flor = 0.025 * escala
    
    # Distribuição das flores pelo arbusto
    flores_pos = [
        (-0.05, 0.02, cor_flor_vermelho),
        (0.06, -0.01, cor_flor_amarelo),
        (-0.02, 0.07, cor_flor_roxo),
        (0.05, 0.02, cor_flor_vermelho),
        (-0.03, -0.02, cor_flor_roxo),
    ]
    
    for fx, fy, fcor in flores_pos:
        desenhar_circulo(x + fx * escala, y + fy * escala, raio_flor * 0.8, fcor)
        # Centro da flor (ponto amarelo)
        desenhar_circulo(x + fx * escala, y + fy * escala, raio_flor * 0.3, (1.0, 1.0, 0.2))
    

#funções que criam os objetos 3D (gato e nuvem)
def desenhar_gato(x, y, angulo):
# Gato na posição (x,y), com uma rotação controlada pelo teclado (vira para os lados)
    mat_gato = multiplica_matriz(mat_translacao(x, y, 0.0), mat_rotacao_y(angulo))

    # corpo do gato 
    glUniformMatrix4fv(loc_transform_3d, 1, GL_TRUE, mat_gato)
    glUniform4f(loc_color_3d, cor_gato[0], cor_gato[1], cor_gato[2], 1.0)
    glBindVertexArray(vao_gato)
    glDrawArrays(GL_TRIANGLES, 0, qtd_vertices_gato)

    # olhos do gato - reaproveitando a esfera (vao_bola) 
    olhos_offsets = [(0.135, 0.045, -0.02), (0.135, 0.045, 0.02)]
    for (ex, ey, ez) in olhos_offsets:
        mat_olho = multiplica_matriz(mat_gato, multiplica_matriz(
            mat_translacao(ex, ey, ez), mat_escala(0.15, 0.15, 0.15)))
        glUniformMatrix4fv(loc_transform_3d, 1, GL_TRUE, mat_olho)
        glUniform4f(loc_color_3d, cor_olhos_gato[0], cor_olhos_gato[1], cor_olhos_gato[2], 1.0)
        glBindVertexArray(vao_bola)
        glDrawElements(GL_TRIANGLES, len(indices_bola), GL_UNSIGNED_INT, None)


def desenhar_nuvem(x, y, angulo=0.0):
# Nuvem na posição (x,y), reaproveitando várias vezes a mesma esfera (vao_bola) 
    mat_nuvem = multiplica_matriz(mat_translacao(x, y, 0.0), mat_rotacao_y(angulo))
    for (ox, oy, oz, escala) in puffs_nuvem:
        mat_puff = multiplica_matriz(mat_nuvem, multiplica_matriz(
            mat_translacao(ox, oy, oz), mat_escala(escala, escala, escala)))
        glUniformMatrix4fv(loc_transform_3d, 1, GL_TRUE, mat_puff)
        glUniform4f(loc_color_3d, cor_nuvem[0], cor_nuvem[1], cor_nuvem[2], 1.0)
        glBindVertexArray(vao_bola)
        glDrawElements(GL_TRIANGLES, len(indices_bola), GL_UNSIGNED_INT, None)


In [18]:
#definindo as cores

azul_ceu = (0.4, 0.6, 0.9)
verde_grama = (0.2, 0.6, 0.2)
cor_tronco = (0.4, 0.2, 0.1)
cor_copa = (0.1, 0.5, 0.1)
verde_arbusto = (0.2, 0.6, 0.1)
verde_claro = (0.3, 0.7, 0.2)
cor_flor_vermelho = (1.0, 0.2, 0.3)  # Vermelho/rosa
cor_flor_amarelo = (1.0, 0.8, 0.0)  # Amarelo
cor_flor_roxo = (0.8, 0.2, 0.8)  # Roxo
cor_centro = (1.0, 1.0, 0.0)  # Amarelo claro para o centro

#variáveis de posição da casa
t_x = 0.0
t_y = -0.15 #aqui é a posição inicial da casa pois é ela que vamos transladar e foi a nossa referência

#variaveis para o carro
carro_x = t_x
carro_direcao = 1 # 1 = direita, -1 = esquerda
velocidade_carro = 0.005


#variaveis para a escala do carro
escala_carro = 1.0 #escala "original"
fator_escala = 0.1  # Quanto aumenta a cada pressionamento
escala_maxima = 2.5  # Tamanho máximo
escala_minima = 0.5  # Tamanho mínimo


#variável global para mostrar malha
mostrarMalha = False

In [19]:
#cores e posições dos objetos 3D (gato e nuvem)

cor_gato = (0.85, 0.45, 0.15)         # laranja - corpo do gato
cor_olhos_gato = (0.55, 0.85, 0.20)   # verde - olhos do gato
cor_nuvem = (0.95, 0.95, 0.98)        # branco - nuvem

pos_gato = (0.35, -0.85, 0.0)    # posição do gato no jardim
pos_nuvem = (-0.55, 0.72, 0.0)   # posição da nuvem no céu
angulo_nuvem = 0.0   # ângulo de rotação da nuvem

In [20]:
"""matrizes que vamos usar"""

#2D

#matriz identidade
mat_identity = np.array([
    1.0, 0.0, 0.0, 0.0,
    0.0, 1.0, 0.0, 0.0,
    0.0, 0.0, 1.0, 0.0,
    0.0, 0.0, 0.0, 1.0
], np.float32)

#matriz de transformação
glUniformMatrix4fv(loc, 1, GL_TRUE, mat_identity)

#3D
# Matrizes de projeção e visão para 3D
width, height = 720, 600
projection3D = glm.perspective(glm.radians(45.0), width/height, 0.1, 10.0)
view3D = glm.lookAt(glm.vec3(0.0, 0.0, 0.8), glm.vec3(0.0, 0.0, 0.0), glm.vec3(0.0, 1.0, 0.0))

In [21]:
# Matrizes auxiliares (translação, escala, rotação e composição) para posicionar e transformar o gato e a nuvem. 
def mat_translacao(tx, ty, tz=0.0):
    return np.array([
        1.0, 0.0, 0.0, tx,
        0.0, 1.0, 0.0, ty,
        0.0, 0.0, 1.0, tz,
        0.0, 0.0, 0.0, 1.0], np.float32)

def mat_escala(sx, sy, sz=1.0):
    return np.array([
        sx,  0.0, 0.0, 0.0,
        0.0, sy,  0.0, 0.0,
        0.0, 0.0, sz,  0.0,
        0.0, 0.0, 0.0, 1.0], np.float32)

def mat_rotacao_y(theta):
    c, s = math.cos(theta), math.sin(theta)
    return np.array([
         c,  0.0,  s,  0.0,
        0.0, 1.0, 0.0, 0.0,
        -s,  0.0,  c,  0.0,
        0.0, 0.0, 0.0, 1.0], np.float32)

def multiplica_matriz(a, b):
    """Composição de transformações: M = A . B"""
    m_a = a.reshape(4, 4)
    m_b = b.reshape(4, 4)
    m_c = np.dot(m_a, m_b)
    return m_c.reshape(16).astype(np.float32)


# localização das variáveis uniformes do elemento 3D 
loc_transform_3d = glGetUniformLocation(program_3d, "mat_transformation")
loc_color_3d = glGetUniformLocation(program_3d, "objectColor")

In [22]:
#teclas de evento
def key_event(window,key,scancode,action,mods):
    global t_x, t_y, escala_carro, angulo_gato, mostrarMalha 

    #controle da velocidade do movimento
    velocidade = 0.005
    
    # Verifica se a tecla foi pressionada ou está sendo segurada
    if action == glfw.PRESS or action == glfw.REPEAT:
        # Movimento da casa com as setas
        if key == 265: t_y += velocidade  # cima
        if key == 264: t_y -= velocidade  # baixo
        if key == 263: t_x -= velocidade  # esquerda
        if key == 262: t_x += velocidade  # direita
        
        if key == 65 or key == 97:  # Código da tecla a minuscula
            escala_carro += fator_escala
            if escala_carro > escala_maxima:
                escala_carro = escala_maxima
        
        if key == 100 or key == 68:  # Código da tecla d minuscula
            escala_carro -= fator_escala
            if escala_carro < escala_minima:
                escala_carro = escala_minima

         # Rotação do gato 
        if key == 81 or key == 113:  # tecla q - gira para um lado
            angulo_gato += 0.1
        if key == 69 or key == 101:  # tecla e - gira para o outro lado
            angulo_gato -= 0.1

        #malha 
        if key == 80 or key == 112:  # Tecla m (maiúscula ou minúscula)
            mostrarMalha = not mostrarMalha

glfw.set_key_callback(window,key_event)

In [23]:
#exibindo a janela

glfw.show_window(window)

In [24]:
#loop principal 
tempo_nuvem = 0.0
angulo_gato = 0.0

while not glfw.window_should_close(window):
    # Limpa a tela
    glClear(GL_COLOR_BUFFER_BIT)
    glClearColor(1.0, 1.0, 1.0, 1.0)

    # Movimento do carro
    carro_x += velocidade_carro * carro_direcao
    
    # Inverte a direção quando o carro chega nas bordas (considerando a escala)
    limite = 0.8 - (0.15 * escala_carro)  # Quanto maior o carro, mais cedo ele vira
    if carro_x > limite:
        carro_direcao = -1
    elif carro_x < -limite:
        carro_direcao = 1

    if mostrarMalha:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
        glLineWidth(1.5)  # Linhas mais grossas para melhor visualização
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)
    
    #PARTE 2D
    glUseProgram(program)
    glBindVertexArray(0)
    glUniformMatrix4fv(loc_transform, 1, GL_TRUE, mat_identity)
    
    # Céu
    desenhar_retangulo(-1.0, 0.0, 2.0, 1.0, azul_ceu)
    
    # Chão
    desenhar_retangulo(-1.0, -1.0, 2.0, 1.0, verde_grama)
    
    # Casa
    desenhar_casa(t_x, t_y)
    
    # baloes
    desenhar_baloes(t_x, t_y + 0.50)  # Posicionado no topo da casa
    
    # Árvores
    desenhar_arvore(-0.6, -0.16, cor_tronco, cor_copa)
    desenhar_arvore(0.6, -0.16, cor_tronco, cor_copa)
    desenhar_arvore(-0.4, -0.26, cor_tronco, cor_copa)
    desenhar_arvore(0.4, -0.21, cor_tronco, cor_copa)
    desenhar_arvore(-0.8, -0.31, cor_tronco, cor_copa)
    desenhar_arvore(0.8, -0.34, cor_tronco, cor_copa)

    #arbustros
    desenhar_arbusto_com_flores(-0.6, -0.42, 1.2)
    desenhar_arbusto_com_flores(0.6, -0.42, 1.2)
    desenhar_arbusto_com_flores(-0.4, -0.82, 1.2)
    desenhar_arbusto_com_flores(0.5, -0.72, 1.2)
    desenhar_arbusto_com_flores(0, -0.42, 1.2)

    
    # carro
    desenhar_carro(carro_x, -0.65, escala_carro)

    # 3D (gato e nuvem)
    tempo_nuvem += 0.015
    angulo_nuvem = math.sin(tempo_nuvem) * 0.35
    glUseProgram(program_3d)
    desenhar_gato(pos_gato[0], pos_gato[1], angulo_gato)
    desenhar_nuvem(pos_nuvem[0], pos_nuvem[1], angulo_nuvem)
    
    # Atualiza a tela
    glfw.swap_buffers(window)
    glfw.poll_events()

glfw.terminate()